In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

demo = r"C:\Users\User\Documents\Iron Hack\Week5\Project\Project-2-Vanguard-CX-Team\Data\Raw Data\df_final_demo.txt"
exp = r"C:\Users\User\Documents\Iron Hack\Week5\Project\Project-2-Vanguard-CX-Team\Data\Raw Data\df_final_experiment_clients.txt"
webdata1 = r"C:\Users\User\Documents\Iron Hack\Week5\Project\Project-2-Vanguard-CX-Team\Data\Raw Data\df_final_web_data_pt_1.txt"
webdata2 = r"C:\Users\User\Documents\Iron Hack\Week5\Project\Project-2-Vanguard-CX-Team\Data\Raw Data\df_final_web_data_pt_2.txt"
merged = r"C:\Users\User\Documents\Iron Hack\Week5\Project\Project-2-Vanguard-CX-Team\Data\Clean Data\merged_data_with_NaN.csv"
analysis = r"C:\Users\User\Documents\Iron Hack\Week5\Project\Project-2-Vanguard-CX-Team\Data\Clean Data\analysis.csv"


exp_df = pd.read_csv(exp)
demo_df = pd.read_csv(demo)
webdata1_df = pd.read_csv(webdata1)
webdata2_df = pd.read_csv(webdata2)
merged_df = pd.read_csv(merged)
analysis_df = pd.read_csv(analysis)

C:\Users\User\AppData\Local\Temp\ipykernel_30288\1722363511.py:19: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv(merged)


In [2]:
type ('date_time')

str

In [3]:
analysis_df

,client_id,visitor_id,visit_id,process_step,date_time,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317230,1574008,117364417_77840596075,528720790_71583064618_169151,start,2017-05-06 23:43:27,Test,10.0,121.0,55.0,U,2.0,153238.83,3.0,6.0
317231,2908510,814969699_90652851448,562606085_36368381773_92090,start,2017-05-10 22:57:17,Control,21.0,252.0,34.0,M,3.0,141808.05,6.0,9.0
317232,2908510,814969699_90652851448,562606085_36368381773_92090,step_2,2017-05-10 22:56:31,Control,21.0,252.0,34.0,M,3.0,141808.05,6.0,9.0
317233,2908510,814969699_90652851448,562606085_36368381773_92090,step_1,2017-05-10 22:56:23,Control,21.0,252.0,34.0,M,3.0,141808.05,6.0,9.0


In [4]:
# Convert 'date_time' to datetime, coercing errors
analysis_df['date_time'] = pd.to_datetime(analysis_df['date_time'], errors='coerce')

In [10]:
print("KPI 1: COMPLETION RATE")
print("Definition: Proportion of users who reach the final 'confirm' step\n")

# Count total unique clients per variation 
total_clients = analysis_df.groupby('Variation')['client_id'].nunique()

#Count clients who reached 'confirmed' step 
completed_clients = analysis_df[analysis_df['process_step'] == 'confirm'].groupby('Variation')['client_id'].nunique()

#calculate completion rate
completion_rate = (completed_clients / total_clients * 100).round(2)

# Display results
print("Total Clients:")
print(total_clients)
print("\nClients Who Completed:")
print(completed_clients)
print("\nCOMPLETION RATE:")
for var in completion_rate.index:
    print(f"  {var:12s}: {completion_rate[var]:6.2f}%")

# Calculate improvement
if 'Test' in completion_rate.index and 'Control' in completion_rate.index:
    abs_diff = completion_rate['Test'] - completion_rate['Control']
    rel_diff = ((completion_rate['Test'] / completion_rate['Control']) - 1) * 100
    
    print(f"\n🎯 IMPROVEMENT:")
    print(f"  Absolute: {abs_diff:+.2f} percentage points")
    print(f"  Relative: {rel_diff:+.2f}%")
    
    if abs_diff > 0:
        print(f"  ✅ Test group shows HIGHER completion rate")
    elif abs_diff < 0:
        print(f"  ❌ Test group shows LOWER completion rate")
    else:
        print(f"  ➖ No difference in completion rate")




# ============================================================================
# KPI 2: TIME SPENT ON EACH STEP
# ============================================================================

print("\n" + "="*80)
print("KPI 2: TIME SPENT ON EACH STEP")
print("="*80)
print("Definition: Average duration users spend on each step\n")

# Sort data by client, visit, and timestamp
df_sorted = analysis_df.sort_values(['client_id', 'visit_id', 'date_time']).copy()

# Calculate time difference between consecutive steps within same visit
df_sorted['time_diff'] = df_sorted.groupby(['client_id', 'visit_id'])['date_time'].diff()

# Convert to seconds then minutes
df_sorted['time_diff_seconds'] = df_sorted['time_diff'].dt.total_seconds()
df_sorted['time_diff_minutes'] = df_sorted['time_diff_seconds'] / 60

# Filter outliers: Remove sessions > 60 minutes (likely abandoned/interrupted)
df_sorted.loc[df_sorted['time_diff_minutes'] > 60, 'time_diff_minutes'] = np.nan

# Calculate average time per step by variation
time_per_step = df_sorted.groupby(['Variation', 'process_step'])['time_diff_minutes'].agg([
    ('mean', 'mean'),
    ('median', 'median'),
    ('count', 'count')
]).round(2)

print("📊 AVERAGE TIME PER STEP (minutes):")
print(time_per_step)


# Calculate total journey time
total_journey = df_sorted.groupby(['Variation', 'client_id', 'visit_id'])['time_diff_minutes'].sum()
avg_total_time = total_journey.groupby('Variation').agg(['mean', 'median', 'std']).round(2)

print("\n📊 AVERAGE TOTAL JOURNEY TIME (minutes):")
print(avg_total_time)

# Compare Control vs Test
if 'Test' in avg_total_time.index and 'Control' in avg_total_time.index:
    time_diff = avg_total_time.loc['Test', 'mean'] - avg_total_time.loc['Control', 'mean']
    time_pct = (time_diff / avg_total_time.loc['Control', 'mean']) * 100
    
    print(f"\n🎯 JOURNEY TIME COMPARISON:")
    print(f"  Difference: {time_diff:+.2f} minutes ({time_pct:+.1f}%)")
    if time_diff < 0:
        print(f"  ✅ Test group is FASTER")
    elif time_diff > 0:
        print(f"  ❌ Test group is SLOWER")
    else:
        print(f"  ➖ No time difference")


# ============================================================================
# KPI 3: ERROR RATE (Backward Navigation)
# ============================================================================

print("\n" + "="*80)
print("KPI 3: ERROR RATE (Backward Navigation)")
print("="*80)
print("Definition: Moving from a later step to an earlier step indicates confusion\n")

# Define step order
step_order = {
    'start': 0,
    'step_1': 1,
    'step_2': 2,
    'step_3': 3,
    'confirm': 4
}

# Map steps to numbers
df_sorted['step_number'] = df_sorted['process_step'].map(step_order)

# Calculate step change (difference between consecutive steps)
df_sorted['step_change'] = df_sorted.groupby(['client_id', 'visit_id'])['step_number'].diff()

# Identify backward navigation (negative step change = error)
df_sorted['is_backward'] = df_sorted['step_change'] < 0

# Calculate error rate by variation
total_transitions = df_sorted[df_sorted['step_change'].notna()].groupby('Variation').size()
backward_transitions = df_sorted[df_sorted['is_backward'] == True].groupby('Variation').size()
error_rate = (backward_transitions / total_transitions * 100).round(2)

print("📊 STEP TRANSITIONS:")
print(f"Total transitions:")
print(total_transitions)
print(f"\nBackward navigations (errors):")
print(backward_transitions)

print("\n📊 ERROR RATE:")
for var in error_rate.index:
    print(f"  {var:12s}: {error_rate[var]:6.2f}%")

# Calculate improvement
if 'Test' in error_rate.index and 'Control' in error_rate.index:
    error_diff = error_rate['Test'] - error_rate['Control']
    
    print(f"\n🎯 ERROR RATE COMPARISON:")
    print(f"  Difference: {error_diff:+.2f} percentage points")
    if error_diff < 0:
        print(f"  ✅ Test group has FEWER errors")
    elif error_diff > 0:
        print(f"  ❌ Test group has MORE errors")
    else:
        print(f"  ➖ No difference in error rate")

# Show where errors happen most
print("\n📊 ERRORS BY STEP:")
errors_by_step = df_sorted[df_sorted['is_backward'] == True].groupby(['Variation', 'process_step']).size()
print(errors_by_step)

#============================================================================
# KPI 4: DROP-OFF RATE (Additional Important KPI)
# ============================================================================

print("\n" + "="*80)
print("KPI 4: DROP-OFF RATE (Additional KPI)")
print("="*80)
print("Definition: % of users who don't proceed to the next step\n")

# Count users at each step
users_per_step = analysis_df.groupby(['Variation', 'process_step'])['client_id'].nunique().unstack(fill_value=0)

print("📊 USERS AT EACH STEP:")
print(users_per_step)

# Calculate drop-off between consecutive steps
steps = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

print("\n📊 DROP-OFF RATES:")
for var in users_per_step.index:
    print(f"\n{var} Group:")
    for i in range(len(steps) - 1):
        current = steps[i]
        next_step = steps[i + 1]
        
        if current in users_per_step.columns and next_step in users_per_step.columns:
            current_users = users_per_step.loc[var, current]
            next_users = users_per_step.loc[var, next_step]
            
            if current_users > 0:
                drop_rate = ((current_users - next_users) / current_users * 100)
                retention_rate = (next_users / current_users * 100)
                print(f"  {current:8s} → {next_step:8s}: {drop_rate:5.1f}% drop-off ({retention_rate:5.1f}% retained)")



# ============================================================================
# KPI 5: ENGAGEMENT DEPTH (Additional KPI)
# ============================================================================

print("\n" + "="*80)
print("KPI 5: ENGAGEMENT DEPTH (Additional KPI)")
print("="*80)
print("Definition: How far users progress through the funnel\n")

# Calculate max step reached per client
max_step = df_sorted.groupby(['Variation', 'client_id'])['step_number'].max()
avg_max_step = max_step.groupby('Variation').agg(['mean', 'median']).round(2)

print("📊 AVERAGE MAX STEP REACHED (0=start, 4=confirm):")
print(avg_max_step)

# Step distribution
step_distribution = (max_step.groupby(['Variation', max_step]).size() / 
                     max_step.groupby('Variation').size() * 100).unstack(fill_value=0).round(1)

print("\n📊 DISTRIBUTION OF MAX STEP REACHED (%):")
print(step_distribution)


# ============================================================================
# KPI 6: VISIT EFFICIENCY (Additional KPI)
# ============================================================================

print("\n" + "="*80)
print("KPI 6: VISIT EFFICIENCY (Additional KPI)")
print("="*80)
print("Definition: Average number of visits needed per client\n")

visits_per_client = analysis_df.groupby(['Variation', 'client_id'])['visit_id'].nunique()
avg_visits = visits_per_client.groupby('Variation').agg(['mean', 'median', 'std']).round(2)

print("📊 AVERAGE VISITS PER CLIENT:")
print(avg_visits)


# ============================================================================
# COMPREHENSIVE KPI SUMMARY TABLE
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE KPI SUMMARY")
print("="*80)

# Create summary DataFrame
kpi_summary = pd.DataFrame({
    'Completion Rate (%)': completion_rate,
    'Avg Journey Time (min)': avg_total_time['mean'],
    'Error Rate (%)': error_rate,
    'Avg Max Step (0-4)': avg_max_step['mean'],
    'Avg Visits per Client': avg_visits['mean']
})

print("\n" + kpi_summary.to_string())

# Calculate improvements (Test vs Control)
if 'Test' in kpi_summary.index and 'Control' in kpi_summary.index:
    print("\n" + "="*80)
    print("🎯 TEST vs CONTROL COMPARISON (Improvement)")
    print("="*80)
    
    improvements = pd.DataFrame({
        'Control': kpi_summary.loc['Control'],
        'Test': kpi_summary.loc['Test'],
        'Absolute Diff': kpi_summary.loc['Test'] - kpi_summary.loc['Control'],
        'Relative Diff (%)': ((kpi_summary.loc['Test'] / kpi_summary.loc['Control']) - 1) * 100
    })
    
    print("\n" + improvements.round(2).to_string())



KPI 1: COMPLETION RATE
Definition: Proportion of users who reach the final 'confirm' step

Total Clients:
Variation
Control    23532
Test       26968
Name: client_id, dtype: int64

Clients Who Completed:
Variation
Control    15434
Test       18687
Name: client_id, dtype: int64

COMPLETION RATE:
  Control     :  65.59%
  Test        :  69.29%

🎯 IMPROVEMENT:
  Absolute: +3.70 percentage points
  Relative: +5.64%
  ✅ Test group shows HIGHER completion rate

KPI 2: TIME SPENT ON EACH STEP
Definition: Average duration users spend on each step

📊 AVERAGE TIME PER STEP (minutes):
                        mean  median  count
Variation process_step                     
Control   confirm       2.15    1.22  16570
          start         2.73    0.85  14455
          step_1        0.72    0.30  29258
          step_2        0.65    0.33  25659
          step_3        1.57    1.10  22329
Test      confirm       2.13    0.95  22204
          start         2.43    0.95  22694
          step_1       

In [13]:
# Create summary DataFrame
kpi_summary = pd.DataFrame({
    'Completion Rate (%)': completion_rate,
    'Avg Journey Time (min)': avg_total_time['mean'],
    'Error Rate (%)': error_rate,
    'Avg Max Step (0-4)': avg_max_step['mean'],
    'Avg Visits per Client': avg_visits['mean']
})

print("\n" + kpi_summary.to_string())



# Create ExcelWriter object
file_path = "kpi_analysis_full.xlsx"
with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    kpi_summary.to_excel(writer, sheet_name='kpi_summary', index=False)


print(f"KPIs and analysis have been successfully exported to {file_path}")


           Completion Rate (%)  Avg Journey Time (min)  Error Rate (%)  Avg Max Step (0-4)  Avg Visits per Client
Variation                                                                                                        
Control                  65.59                    4.58            8.85                3.06                   1.37
Test                     69.29                    5.18           11.64                3.22                   1.38
KPIs and analysis have been successfully exported to kpi_analysis_full.xlsx
